### Context Engineering In Deep Agents

In [ ]:
import os
from pathlib import Path
from urllib.request import urlopen
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from deepagents import create_deep_agent
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from deepagents.backends.store import StoreBackend
from deepagents.backends import StateBackend
from deepagents.backends.filesystem import FilesystemBackend
from deepagents.backends.utils import create_file_data
from langchain_quickjs import CodeInterpreterMiddleware

load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [ ]:
# Model Initialization
llm = ChatGroq(model="openai/gpt-oss-120b", api_key=groq_api_key, max_tokens=1000)

#### Context Engineering - Input Context (System Prompt)

In [ ]:
agent = create_deep_agent(
    model=llm,
    system_prompt="You are a research assistant specializing in scientific literature. ",
)

agent

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is ml?"}]})
result

#### Context Engineering - Input Context (Memory - AGENTS.md)

In [ ]:
agents_md = Path("memory/AGENTS.md").read_text(encoding="utf-8")
agents_md

In [ ]:
# Ways to Inject AGENTS.md file content:
# When using Checkpointer (InMemorySaver)

checkpoint = InMemorySaver()

agent = create_deep_agent(
    model=llm,
    checkpointer=checkpoint,  # no backend, no memory=, no files=
)
result = agent.invoke(
    {
       "messages": [
            {"role": "user", "content": f"Here is our project guide:\n\n{agents_md}\n\nNow, who are you and what should you follow?"}
        ]
    },
    config={"configurable": {"thread_id": "default-demo-1"}},  # fresh thread => memory loads
)
print(result["messages"][-1].content)

In [ ]:
agent=create_deep_agent(
    model=llm,
    memory=["/memory/AGENTS.md"],
    checkpointer=InMemorySaver(),
)

In [ ]:
create_file_data(agents_md)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "What's in your memory? Who are you?"}],
        "files": {"/projects/AGENTS.md": create_file_data(agents_md)},  # seed the in-state file
    },
    config={"configurable": {"thread_id": "default-demo-1"}},  # fresh thread => memory loads (only for the first time call)
)
print(result["messages"][-1].content)

In [ ]:
store = InMemoryStore() # Long-term memory (cross-thread, per session)
checkpointer = InMemorySaver() # Short-term memory (single-thread, per session)

# Seed durable memory into the store once. Namespace + key identify the file.
store.put(("memories",), "AGENTS.md", create_file_data(agents_md))

store_agent = create_deep_agent(
    model=llm,
    backend=StoreBackend(store=store, namespace=lambda rt: ("memories",)),
    memory=["/memory/AGENTS.md"],            # read from the store, not state or disk
    store=store,
    checkpointer=checkpointer,
)

# Note: no files= seeding needed — the store already holds it, and it would persist
# even if we started a brand-new thread or rebuilt the agent but not the kernel
result = store_agent.invoke(
    {"messages": [{"role": "user", "content": "What's in your memory? Who are you?"}]},
    config={"configurable": {"thread_id": "store-demo-1"}},
)

print(result["messages"][-1].content)

In [ ]:
fs_agent = create_deep_agent(
    model=llm,
    backend=FilesystemBackend(root_dir="memory", virtual_mode=True),
    memory=["AGENTS.md"],                             # -> memory/AGENTS.md
    checkpointer=InMemorySaver(),
)

# Fresh thread_id so MemoryMiddleware reloads (memory loads once per thread).
result = fs_agent.invoke(
    {"messages": [{"role": "user", "content": "What's in your memory? Who are you?"}]},
    config={"configurable": {"thread_id": "fs-demo-2"}},
)

print(result["messages"][-1].content)

#### Context Engineering - Input Context (Skills)

In [ ]:
checkpointer = InMemorySaver()
backend = StateBackend()

# To load a single skill
skills_content= Path("skills/langgraph/SKILL.md").read_text(encoding="utf-8")

skills_files = {
    "/skills/langgraph/SKILL.md": create_file_data(skills_content),
}

In [ ]:
# Way to load multiple skills
skill_dirs = ["langgraph", "python", "aws", "report-writer"]

skills_files = {
    f"/skills/{name}/SKILL.md": create_file_data(
        Path(f"skills/{name}/SKILL.md").read_text(encoding="utf-8")
    )
    for name in skill_dirs
}

In [ ]:
skills_files

In [ ]:
agent=create_deep_agent(
    model=llm,
    backend=backend,
    skills=["/skills/"],
    checkpointer=checkpointer
)

result = agent.invoke(
    {
        "messages": [{"role": "user", "content": "What skills do you have available, and when would you use each?"}],
        
        "files": skills_files,
    },
    config={"configurable": {"thread_id": "skills-demo1"}},
)

print(result["messages"][-1].content)

In [ ]:
result

In [ ]:
result = agent.invoke(
      {
          "messages": [{
              "role": "user",
              "content": "How do I build a LangGraph graph with conditional routing and memory? Show a minimal example.",       
          }],
          "files": skills_files,   # seed skills into THIS thread's state
      },
      config={"configurable": {"thread_id": "skills-demo-2"}},
  )

# Print the conversation so you can SEE the skill being triggered:
# look for a ToolMessage from `read_file` on the langgraph-docs SKILL.md.
for m in result["messages"]:
    m.pretty_print()

In [ ]:
result = agent.invoke(
      {
          "messages": [{
              "role": "user",
              "content": "write me a python code to do binary search",       
          }],
          "files": skills_files,   # seed skills into THIS thread's state
      },
      config={"configurable": {"thread_id": "skills-demo-4"}},
  )

# Print the conversation so you can SEE the skill being triggered:
# look for a ToolMessage from `read_file` on the langgraph-docs SKILL.md.
for m in result["messages"]:
    m.pretty_print()